# Dataset Aggregation, Filtering & Merging

## Workflow Order

The order matters a lot here. Here is what we do and why:

1. **Preprocessing** — fix dates, remove residual duplicates, handle known data issues
2. **Billings — keep only the most recent record per customer** — billings has multiple rows per co_ref (one per renewal year). We keep only the latest.
3. **Filter billings to Won/Churned only** — Open/unknown outcomes cannot train a churn model
4. **Emails — aggregate duplicates** — emails has 3,211 co_refs with 2 rows. Aggregate first.
5. **Filter using emails (14_out)** — emails is pre-filtered to 14_out. Use its co_refs to filter billings. This gives us only customers who had an email interaction 14 days before renewal.
6. **Renewal calls — apply 45-to-14 day window, then aggregate** — filter calls to the window, then collapse to one row per customer
7. **CC calls — aggregate** — multiple calls per customer, collapse to one row
8. **Merge all four datasets** — left join on co_ref from billings (base)

In [ ]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', None)

---
## Step 1 — Load All Datasets

In [ ]:
billings      = pd.read_csv('billings_cleaned.csv',      low_memory=False)
renewal_calls = pd.read_csv('renewal_calls_cleaned.csv', low_memory=False)
emails        = pd.read_csv('emails_cleaned.csv',        low_memory=False)
cc_calls      = pd.read_csv('cc_calls_cleaned.csv',      low_memory=False)

print(f'billings:      {billings.shape}')
print(f'renewal_calls: {renewal_calls.shape}')
print(f'emails:        {emails.shape}')
print(f'cc_calls:      {cc_calls.shape}')

---
## Step 2 — Preprocessing (Fix Issues Found During Exploration)

In [ ]:
# ── 2a. Convert date columns to proper datetime ───────────────────────────────
# All date columns are stored as strings and need to be datetime for
# the 45-to-14 day window calculation to work correctly

billings['prospect_renewal_date'] = pd.to_datetime(
    billings['prospect_renewal_date'], errors='coerce'
)
billings['registration_date'] = pd.to_datetime(
    billings['registration_date'], errors='coerce'
)
billings['closed_date'] = pd.to_datetime(
    billings['closed_date'], errors='coerce'
)

renewal_calls['call_date'] = pd.to_datetime(
    renewal_calls['call_date'], errors='coerce'
)

cc_calls['call_date'] = pd.to_datetime(
    cc_calls['call_date'], errors='coerce'
)

# Drop rows where call_date parsed as NaT — cannot use them in window filter
rc_before = len(renewal_calls)
renewal_calls = renewal_calls.dropna(subset=['call_date'])
print(f'renewal_calls: dropped {rc_before - len(renewal_calls)} rows with null call_date')

cc_before = len(cc_calls)
cc_calls = cc_calls.dropna(subset=['call_date'])
print(f'cc_calls: dropped {cc_before - len(cc_calls)} rows with null call_date')

In [ ]:
# ── 2b. Remove residual duplicates ───────────────────────────────────────────
# renewal_calls and cc_calls still had a small number of exact duplicate rows

before = len(renewal_calls)
renewal_calls = renewal_calls.drop_duplicates()
print(f'renewal_calls: removed {before - len(renewal_calls)} exact duplicates')

before = len(cc_calls)
cc_calls = cc_calls.drop_duplicates()
print(f'cc_calls: removed {before - len(cc_calls)} exact duplicates')

In [ ]:
# ── 2c. Convert Yes/No columns in cc_calls to binary (1/0) ───────────────────
# These columns only have 'Yes' and 'No' values.
# Converting to 1/0 now makes the max() aggregation meaningful later.
# (max of 1/0 = 1 if it ever happened, 0 if it never did)

cc_binary_cols = [
    'cc_care_package_discussed', 'cc_urgency_getting_on_site',
    'cc_external_consultant', 'cc_agent_cross_sell_attempt',
    'cc_customer_issues_concerns', 'cc_business_struggles_financial_hardship',
    'cc_questionnaire_completion', 'cc_chasing_response',
    'cc_issues_within_questionnaire', 'cc_login_issues', 'cc_platform_issues',
    'cc_dissatisfaction_time_to_complete', 'cc_process_complexity_concerns',
    'cc_questions_harder_than_expected', 'cc_dissatisfaction_support',
    'cc_pricing_mentioned', 'cc_pricing_sentiment_impact',
    'cc_refund_discussed', 'cc_contractor_suggest_leave', 'cc_contractor_complained',
]

for col in cc_binary_cols:
    cc_calls[col] = (cc_calls[col].astype(str).str.strip() == 'Yes').astype(int)

print(f'Converted {len(cc_binary_cols)} Yes/No columns in cc_calls to binary.')

In [ ]:
# ── 2d. crm_contractor_sentiment_score in emails ─────────────────────────────
# This column contains text labels ('Satisfied', 'Neutral', 'Dissatisfied', 'Unknown')
# NOT numeric scores. Map to ordinal numbers for model use.

sentiment_score_map = {
    'Dissatisfied':  0,
    'Neutral':       1,
    'Satisfied':     2,
    'Not Discussed': np.nan,
    'Unknown':       np.nan,
}
emails['crm_contractor_sentiment_score_num'] = (
    emails['crm_contractor_sentiment_score']
    .astype(str).str.strip()
    .map(sentiment_score_map)
)
print('crm_contractor_sentiment_score_num value counts:')
print(emails['crm_contractor_sentiment_score_num'].value_counts(dropna=False))

---
## Step 3 — Billings: Keep Only the Most Recent Record Per Customer

**Why:** Billings has up to 4 rows per `co_ref` — one for each renewal year (2023, 2024, 2025, 2026).
For churn prediction we need the **latest** record. Using an older record would mean we are predicting
on stale features (old band, old score, old outcome). The most recent row represents the customer's
current state.

In [ ]:
print(f'Billings before: {billings.shape}')
print(f'co_refs with multiple rows: {billings["co_ref"].duplicated().sum()}')

# Sort by prospect_renewal_date descending, keep first (most recent)
billings = (
    billings
    .sort_values('prospect_renewal_date', ascending=False)
    .drop_duplicates(subset='co_ref', keep='first')
    .reset_index(drop=True)
)

print(f'Billings after keeping most recent: {billings.shape}')
print(f'renewal_year distribution after dedup:')
print(billings['renewal_year'].value_counts().sort_index())

---
## Step 4 — Billings: Keep Only Won and Churned Outcomes

**Why:** Rows with `prospect_outcome = 'Open'` have an unknown final outcome.
We cannot label them as churned or retained, so they cannot train or evaluate a churn model.
Including them would add noise without adding signal.

In [ ]:
print(f'Billings prospect_outcome before filter:')
print(billings['prospect_outcome'].value_counts())

billings = billings[billings['prospect_outcome'].isin(['Won', 'Churned'])].copy()

# Create the churn target variable
billings['churn'] = (billings['prospect_outcome'] == 'Churned').astype(int)

print(f'\nBillings after removing unknown outcomes: {billings.shape}')
print(f'Churn rate: {billings["churn"].mean():.2%}')

---
## Step 5 — Emails: Aggregate Duplicates (One Row Per co_ref)

**Why there are duplicates:** 3,211 co_refs have 2 email records.
Looking at the data, these are cases where one row has 'Unknown' for sentiment
and another has an actual value. We aggregate to keep all the signal.

**Aggregation strategy for emails:**
- Categorical Yes/No columns → **mode** (most common value): takes the dominant signal across rows
- Numeric sentiment score → **max** (take the most informative non-null value)
- `crm_agent_chase_count` (Low/Medium/High) → **most severe level** seen (max priority)

In [ ]:
# Define email column groups
email_yn_cols = [
    'crm_accreditation_completed', 'crm_timely_completion',
    'crm_progress_towards_accreditation', 'crm_delays_in_accreditation',
    'crm_contractor_suggested_leave', 'crm_contractor_engagement',
    'crm_dts_or_ssip_mentioned', 'crm_customer_payment_intention',
    'crm_competitors_mentioned', 'crm_platform_issues_raised',
    'crm_agent_chased_contractor', 'crm_accreditation_issues',
    'crm_membership_overdue', 'crm_dissatisified_with_renewal_price',
    'crm_customer_complained', 'crm_refund_mentioned',
    'crm_negative_customer_experience', 'crm_dissatisfaction_with_support',
    'crm_financial_hardship_mentioned',
]

# For Yes/No columns: if either row says 'Yes', the aggregated value should be 'Yes'
# Using max() on 'Yes'/'No' string doesn't work, so convert to binary first,
# take max (1 if any row was Yes), then map back to Yes/No
emails_clean = emails.copy()

for col in email_yn_cols:
    emails_clean[f'{col}_bin'] = (
        emails_clean[col].astype(str).str.strip().isin(['Yes', 'yes'])
    ).astype(int)

bin_cols = [f'{c}_bin' for c in email_yn_cols]

# Chase count: convert Low/Medium/High to ordinal for aggregation
chase_map = {'Low': 1, 'Medium': 2, 'High': 3, 'Unknown': np.nan}
emails_clean['chase_ordinal'] = (
    emails_clean['crm_agent_chase_count']
    .astype(str).str.strip().map(chase_map)
)

# Sentiment ordinal: already created in preprocessing

# Aggregate binary flags with max (1 if it ever appeared across rows for that co_ref)
emails_agg_bin = emails_clean.groupby('co_ref')[bin_cols + ['chase_ordinal', 'crm_contractor_sentiment_score_num']].max().reset_index()

# For the original text columns (sentiment text, membership level etc) take mode
# — the most frequently observed value is the most representative
text_cols_email = ['crm_contractor_sentiment', 'crm_membership_level', 'sentiment_category']
def safe_mode(x):
    m = x.dropna()
    m = m[~m.isin(['Unknown','Not Discussed',''])]
    if len(m) == 0:
        return np.nan
    return m.mode().iloc[0]

emails_agg_text = emails_clean.groupby('co_ref')[text_cols_email].agg(safe_mode).reset_index()

# Merge binary + text aggregations
emails_agg = emails_agg_bin.merge(emails_agg_text, on='co_ref', how='left')

# Map chase ordinal back to label
chase_reverse = {1: 'Low', 2: 'Medium', 3: 'High'}
emails_agg['crm_agent_chase_count_agg'] = emails_agg['chase_ordinal'].map(chase_reverse)
emails_agg = emails_agg.drop(columns=['chase_ordinal'])

# Rename binary columns back to original names
rename_bin = {f'{c}_bin': c for c in email_yn_cols}
emails_agg = emails_agg.rename(columns=rename_bin)

print(f'Emails before aggregation: {emails.shape}')
print(f'Emails after aggregation:  {emails_agg.shape}')
print(f'Unique co_refs after agg:  {emails_agg["co_ref"].nunique()}')

---
## Step 6 — Filter: Keep Only Customers Present in Emails (14_out)

**Why this comes before the date window filter:**
The emails dataset is already filtered to `time_to_renewal == '14_out'`, meaning these customers
had an email interaction exactly in the 14-day window before their renewal.
Using this as our customer universe ensures we only keep customers who were actively
engaged in the renewal process — the exact group we want to build a churn model for.

Customers not in emails are excluded from the dataset.

In [ ]:
email_co_refs = set(emails_agg['co_ref'].unique())
print(f'Unique co_refs in emails (14_out): {len(email_co_refs):,}')

# Filter billings to only customers who have an email record
before = len(billings)
billings = billings[billings['co_ref'].isin(email_co_refs)].copy().reset_index(drop=True)
print(f'Billings: {before:,} → {len(billings):,} rows after email filter')
print(f'Churn rate after email filter: {billings["churn"].mean():.2%}')

# Filter renewal_calls to only these customers too
before = len(renewal_calls)
renewal_calls = renewal_calls[renewal_calls['co_ref'].isin(email_co_refs)].copy().reset_index(drop=True)
print(f'renewal_calls: {before:,} → {len(renewal_calls):,} rows after email filter')

# Filter cc_calls to only these customers
before = len(cc_calls)
cc_calls = cc_calls[cc_calls['co_ref'].isin(email_co_refs)].copy().reset_index(drop=True)
print(f'cc_calls: {before:,} → {len(cc_calls):,} rows after email filter')

---
## Step 7 — Renewal Calls: Apply 45-to-14 Day Window Filter

**Why 45-to-14 and not just 14 days:**
We are predicting churn at the **14-day-out** point. So we want call features that were observable
at that moment — calls from 45 to 14 days before the renewal date.

- Calls within the last 14 days (closer to renewal) belong to the pre-renewal window and
  would introduce data from after our prediction point.
- Calls more than 45 days before renewal are too far in the past to be relevant signals.

This filter needs the `prospect_renewal_date` from billings, so we merge the date
temporarily just for the filter, then work with the filtered calls.

In [ ]:
# Bring renewal date into renewal_calls temporarily for the window calculation
renewal_dates = billings[['co_ref', 'prospect_renewal_date']]

renewal_calls_dated = renewal_calls.merge(renewal_dates, on='co_ref', how='inner')
print(f'renewal_calls after inner join with billings dates: {renewal_calls_dated.shape}')

# Apply the 45-to-14 day window
window_start = renewal_calls_dated['prospect_renewal_date'] - pd.Timedelta(days=45)
window_end   = renewal_calls_dated['prospect_renewal_date'] - pd.Timedelta(days=14)

mask = (
    (renewal_calls_dated['call_date'] >= window_start) &
    (renewal_calls_dated['call_date'] <= window_end)
)

renewal_calls_window = renewal_calls_dated[mask].copy()

# Drop the temporarily added renewal date column — no longer needed
renewal_calls_window = renewal_calls_window.drop(columns=['prospect_renewal_date'])

print(f'\nCalls within 45-to-14 day window: {len(renewal_calls_window):,}')
print(f'Unique customers with calls in window: {renewal_calls_window["co_ref"].nunique():,}')

# Days distribution check
days_diff = (
    (renewal_calls_dated.loc[mask, 'prospect_renewal_date'] -
     renewal_calls_dated.loc[mask, 'call_date']).dt.days
)
print(f'Days before renewal — min: {days_diff.min()}, max: {days_diff.max()}, mean: {days_diff.mean():.1f}')

---
## Step 8 — Renewal Calls: Aggregate to One Row Per Customer

**Aggregation strategy:**

| Column type | Method | Reason |
|-------------|--------|--------|
| Binary flags (0/1) | `max` | 1 if the signal appeared on ANY call. A customer who complained once is still a complainer. |
| Call count / numbers | `sum`, `max`, `mean` | Capture volume and depth of engagement |
| `desire_to_cancel_clean` | Severity priority | cancel > not_discussed > renew — the worst signal across all calls wins |
| `customer_response_score` | `min` | Negative experience on any call is the most important signal |
| `call_direction` | `sum` of each type | Counts of inbound vs outbound calls tells us who is initiating contact |
| Category columns (complaint type etc.) | `mode` + count of non-null | Most common category + whether it occurred at all |

In [ ]:
# ── Define column groups ────────────────────────────────────────────────────
flag_cols = [
    'serious_complaint_flag', 'other_complaint_flag', 'price_discussed_flag',
    'price_impact_renewal_flag', 'discount_requested_flag', 'call_reschedule_flag',
    'membership_alert_flag', 'agent_initiated_renewal_flag', 'competitor_mentioned_flag',
    'switching_intent_flag', 'price_switching_flag', 'pct_price_increase_flag',
    'monetary_price_increase_flag', 'customer_asked_justification_flag',
    'discount_offered_flag', 'renewal_confirmed_flag',
]

# ── Call volume features (all rows) ────────────────────────────────────────
call_volume = renewal_calls_window.groupby('co_ref').agg(
    total_calls           = ('call_id', 'count'),
    outbound_calls        = ('call_direction', lambda x: (x == 'Outbound').sum()),
    inbound_calls         = ('call_direction', lambda x: (x == 'Inbound').sum()),
    max_call_number       = ('call_number', 'max'),
    num_analysed_calls    = ('is_analysed', 'sum'),
).reset_index()

print(f'Call volume features: {call_volume.shape}')

In [ ]:
# ── Signal features from analysed calls only ────────────────────────────────
analysed = renewal_calls_window[renewal_calls_window['is_analysed'] == 1].copy()
print(f'Analysed call rows in window: {len(analysed):,}')

# Binary flags: max across all analysed calls
# Reason: if the customer showed this signal on ANY call, it matters
# (e.g., if they mentioned a competitor once, that's a real signal)
flag_agg = analysed.groupby('co_ref')[flag_cols].max().reset_index()
# Also get sum for key flags (how many times it happened adds more granularity)
flag_sum = analysed.groupby('co_ref')[[
    'serious_complaint_flag', 'competitor_mentioned_flag',
    'switching_intent_flag', 'discount_requested_flag'
]].sum().reset_index().rename(columns={
    'serious_complaint_flag':   'serious_complaint_count',
    'competitor_mentioned_flag':'competitor_mention_count',
    'switching_intent_flag':    'switching_intent_count',
    'discount_requested_flag':  'discount_requested_count',
})

# desire_to_cancel: take the most severe signal seen across all calls
# cancel > not_discussed > renew — because even one cancellation desire matters most
desire_priority = {'cancel': 3, 'not_discussed': 2, 'renew': 1}
analysed['desire_priority'] = (
    analysed['desire_to_cancel_clean']
    .astype(str).str.strip()
    .map(desire_priority)
    .fillna(0)
)
desire_agg = (
    analysed.groupby('co_ref')['desire_priority']
    .max()
    .map({3: 'cancel', 2: 'not_discussed', 1: 'renew', 0: np.nan})
    .reset_index()
    .rename(columns={'desire_priority': 'desire_to_cancel_agg'})
)

# customer_response_score: take the minimum (most negative experience)
# Reason: a customer who was Negative on even one call is at higher churn risk
response_agg = (
    analysed.groupby('co_ref')['customer_response_score']
    .min()
    .reset_index()
    .rename(columns={'customer_response_score': 'min_customer_response_score'})
)
# Also take mean for a smoothed view
response_mean = (
    analysed.groupby('co_ref')['customer_response_score']
    .mean()
    .reset_index()
    .rename(columns={'customer_response_score': 'mean_customer_response_score'})
)

# Categorical columns: mode (most common category seen across calls)
# Also create a binary flag for whether any complaint category was present
def safe_mode(x):
    m = x.dropna()
    if len(m) == 0:
        return np.nan
    return m.mode().iloc[0]

cat_rc_cols = [
    'complaint_category', 'customer_reaction_category',
    'agent_renewal_pitch_category', 'customer_renewal_response_category',
    'agent_response_category',
]
cat_agg = analysed.groupby('co_ref')[cat_rc_cols].agg(safe_mode).reset_index()

# Whether a complaint category was ever logged
cat_agg['had_complaint_category'] = cat_agg['complaint_category'].notna().astype(int)

print(f'Aggregations done for analysed calls.')

In [ ]:
# ── Merge all renewal call aggregations together ─────────────────────────────
renewal_agg = call_volume.copy()
for agg_df in [flag_agg, flag_sum, desire_agg, response_agg, response_mean, cat_agg]:
    renewal_agg = renewal_agg.merge(agg_df, on='co_ref', how='left')

# Flag: customer had at least one call in the window
renewal_agg['had_renewal_call'] = 1

print(f'Renewal calls aggregated: {renewal_agg.shape}')
print(f'Unique customers: {renewal_agg["co_ref"].nunique():,}')

---
## Step 9 — CC Calls: Aggregate to One Row Per Customer

**Why no window filter for CC calls:**
CC calls are customer-initiated support interactions. They do not have a direct date relationship
to the renewal date the way renewal calls do. We use the full history of CC interactions
as a proxy for the customer's general satisfaction with the product and support.

**Aggregation strategy:**

| Column type | Method | Reason |
|-------------|--------|--------|
| Binary columns (0/1) | `max` | 1 if the issue ever occurred across any call |
| Sentiment scores (numeric 0-100) | `mean` | Average score is a stable representation of overall sentiment |
| `cc_contractor_sentiment` text | `mode` (priority to Dissatisfied) | Most critical sentiment seen wins |
| Call count | `count` | Volume of CC interactions indicates level of friction |

In [ ]:
# Sentiment priority for text column: Dissatisfied > Neutral > Satisfied > Not Discussed
cc_sent_priority = {'Dissatisfied': 3, 'Neutral': 2, 'Satisfied': 1, 'Not Discussed': 0}
cc_calls['cc_sentiment_priority'] = (
    cc_calls['cc_contractor_sentiment']
    .astype(str).str.strip()
    .map(cc_sent_priority)
    .fillna(0)
)

# Numeric score columns
cc_score_cols = [
    'cc_contractor_sentiment_start_score',
    'cc_contractor_sentiment_end_score',
    'cc_contractor_sentiment_overall_score',
    'cc_contractor_sentiment_issues_score',
]

# Binary columns (already converted in Step 2c)
cc_agg_dict = {'call_date': 'count'}  # total CC calls

for col in cc_binary_cols:
    cc_agg_dict[col] = 'max'  # 1 if ever happened

for col in cc_score_cols:
    cc_agg_dict[col] = 'mean'  # average score across all CC calls

cc_agg_dict['cc_sentiment_priority'] = 'max'  # worst sentiment seen

cc_agg = cc_calls.groupby('co_ref').agg(cc_agg_dict).reset_index()
cc_agg = cc_agg.rename(columns={'call_date': 'num_cc_calls'})

# Map sentiment priority back to label
cc_sent_reverse = {3: 'Dissatisfied', 2: 'Neutral', 1: 'Satisfied', 0: 'Not Discussed'}
cc_agg['cc_worst_sentiment'] = cc_agg['cc_sentiment_priority'].map(cc_sent_reverse)
cc_agg = cc_agg.drop(columns=['cc_sentiment_priority'])

# Flag: had any CC call
cc_agg['had_cc_call'] = 1

print(f'CC calls aggregated: {cc_agg.shape}')
print(f'Unique customers: {cc_agg["co_ref"].nunique():,}')

---
## Step 10 — Merge All Four Datasets

**Merge strategy:**
- **Base:** billings — every customer we want to model is here
- **Emails:** inner join — we already filtered billings to email customers, so this is safe
- **Renewal calls:** left join — not every customer had a call in the 45-to-14 window
- **CC calls:** left join — not every customer contacted customer care

After merging, customers with no renewal call or CC call will have NaN in those columns.
We fill those NaNs with meaningful defaults (0 for counts/flags, 'None' for categories).

In [ ]:
print(f'Billings rows before merge: {len(billings):,}')

# Start with billings as the base
merged = billings.copy()

# Inner join emails — both sides already filtered to same universe
merged = merged.merge(emails_agg, on='co_ref', how='inner')
print(f'After merging emails (inner):  {merged.shape}')

# Left join renewal call aggregations
merged = merged.merge(renewal_agg, on='co_ref', how='left')
print(f'After merging renewal_calls (left): {merged.shape}')

# Left join CC call aggregations
merged = merged.merge(cc_agg, on='co_ref', how='left')
print(f'After merging cc_calls (left): {merged.shape}')

In [ ]:
# ── Fill NaNs introduced by left joins ───────────────────────────────────────

# Customers with no renewal call in the 45-14 window:
# counts → 0 (they had zero calls), flags → 0, text → 'No_Call'
merged['had_renewal_call'] = merged['had_renewal_call'].fillna(0).astype(int)

renewal_count_cols = [
    'total_calls', 'outbound_calls', 'inbound_calls',
    'max_call_number', 'num_analysed_calls',
    'serious_complaint_count', 'competitor_mention_count',
    'switching_intent_count', 'discount_requested_count',
]
for col in renewal_count_cols:
    if col in merged.columns:
        merged[col] = merged[col].fillna(0).astype(int)

for col in flag_cols:
    if col in merged.columns:
        merged[col] = merged[col].fillna(0).astype(int)

merged['desire_to_cancel_agg']     = merged['desire_to_cancel_agg'].fillna('No_Call')
merged['had_complaint_category']   = merged['had_complaint_category'].fillna(0).astype(int)

# Customers with no CC call:
merged['had_cc_call']  = merged['had_cc_call'].fillna(0).astype(int)
merged['num_cc_calls'] = merged['num_cc_calls'].fillna(0).astype(int)

for col in cc_binary_cols:
    if col in merged.columns:
        merged[col] = merged[col].fillna(0).astype(int)

for col in cc_score_cols:
    if col in merged.columns:
        # Fill with the global mean — neutral assumption for customers without CC interactions
        merged[col] = merged[col].fillna(merged[col].mean())

merged['cc_worst_sentiment'] = merged['cc_worst_sentiment'].fillna('No_CC_Call')

print(f'Final merged dataset: {merged.shape}')
print(f'\nChurn rate: {merged["churn"].mean():.2%}')
print(f'Total missing values: {merged.isnull().sum().sum():,}')

---
## Step 11 — Final Check and Summary

In [ ]:
# Verify one row per co_ref
assert merged['co_ref'].duplicated().sum() == 0, 'ERROR: duplicate co_refs in final merged dataset'
print('One row per co_ref: CONFIRMED')

# Churn distribution
print(f'\nFinal dataset shape: {merged.shape}')
print(f'Churned: {merged["churn"].sum():,} ({merged["churn"].mean():.2%})')
print(f'Retained: {(merged["churn"]==0).sum():,} ({(1-merged["churn"].mean()):.2%})')

# Coverage check: how many customers had calls/emails/cc
print(f'\nData source coverage:')
print(f'  Had renewal call (45-14 window): {merged["had_renewal_call"].sum():,} ({merged["had_renewal_call"].mean():.1%})')
print(f'  Had CC call:                     {merged["had_cc_call"].sum():,} ({merged["had_cc_call"].mean():.1%})')

# Any remaining nulls
remaining_nulls = merged.isnull().sum()
remaining_nulls = remaining_nulls[remaining_nulls > 0].sort_values(ascending=False)
if len(remaining_nulls) > 0:
    print(f'\nRemaining null columns ({len(remaining_nulls)}):')
    print(remaining_nulls.head(15).to_string())
else:
    print('\nNo remaining nulls.')

In [ ]:
# Column summary
print('\nFinal column list:')
for i, col in enumerate(merged.columns, 1):
    dtype = merged[col].dtype
    nulls = merged[col].isnull().sum()
    print(f'  {i:3d}. {col:<50} {str(dtype):<10} nulls={nulls}')

---
## Step 12 — Save Final Dataset

In [ ]:
merged.to_csv('merged_final.csv', index=False)
print(f'Saved merged_final.csv')
print(f'Shape: {merged.shape}')
print(f'Churn rate: {merged["churn"].mean():.2%}')

---
## Full Workflow Summary

| Step | Action | Input → Output | Reason |
|------|--------|----------------|--------|
| 2a | Parse dates | string → datetime | Required for 45-14 window arithmetic |
| 2b | Remove residual duplicates | renewal_calls −2, cc_calls −7 | Exact duplicates inflate call counts |
| 2c | Convert cc Yes/No → binary | string → int | Enables max() aggregation |
| 2d | Map email sentiment score → ordinal | string → 0/1/2 | Makes the text column numeric and usable |
| 3 | Billings dedup → latest row | 113,766 → ~46k rows | Multiple years per customer — only the latest matters for churn prediction |
| 4 | Keep Won/Churned only | Remove Open rows | Cannot label unknown outcomes |
| 5 | Aggregate emails | One row per co_ref (max for binary, mode for text) | 3,211 customers had 2 email rows |
| 6 | Filter by email co_refs | Reduces all datasets to 14_out universe | Emails are the pre-filter gate |
| 7 | Apply 45-to-14 day window | Full renewal calls → window only calls | Only pre-14-day signals are valid at prediction time |
| 8 | Aggregate renewal calls | max for flags, min for response, severity for desire_to_cancel, mode for categories | One row per customer, preserving worst-case signals |
| 9 | Aggregate CC calls | max for binary, mean for scores, severity for sentiment | One row per customer |
| 10 | Merge all four | billings (base) → inner emails → left renewal → left CC | All customers in emails, not all had calls/CC |
| 10b | Fill NaN from left joins | 0 for counts/flags, 'No_Call' for text | Missing = no activity in that channel |